In [7]:
# !pip uninstall torch

# !pip install unsloth trl peft accelerate bitsandbytes torch==2.6

In [8]:
# !pip freeze > colab_requirements.txt

# !python --version


In [9]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
print(f"CUDA device capability: {torch.cuda.get_device_capability(0)}")
print(f"CUDA version: {torch.version.cuda}")

CUDA available: True
CUDA device count: 1
CUDA device name: NVIDIA GeForce RTX 3060
CUDA device capability: (8, 6)
CUDA version: 12.8


In [10]:
from unsloth import FastLanguageModel

model_name = "unsloth/Phi-3.5-mini-instruct-bnb-4bit"
max_seq_length=1024
dtype=None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    dtype=dtype,
    max_seq_length=max_seq_length,
    load_in_4bit=True)



==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
from datasets import load_dataset

DATASET_NAME = "iamtarun/code_contest_python3_alpaca"

ds = load_dataset(DATASET_NAME)  # returns a DatasetDict if splits exist
# If it is a single split dataset, ds may be a Dataset instead.
train_ds = ds["train"] if isinstance(ds, dict) and "train" in ds else (ds if not isinstance(ds, dict) else ds[list(ds.keys())[0]])

# Optional: quick smoke run similar to your current notebook
# train_ds = train_ds.shuffle(seed=3407).select(range(500))


In [12]:
print(train_ds)
print(f"Number of examples: {len(train_ds)}")

# Print features of the dataset
print("Dataset Features:")
for feature, dtype in train_ds.features.items():
    print(f"- {feature}: {dtype}")

# Display the first example
print("\nFirst example from the dataset:")
print(train_ds[0])

IterableDataset({
    features: ['id', 'description', 'code', 'test_samples', 'source', 'prompt'],
    num_shards: 4
})


TypeError: object of type 'IterableDataset' has no len()

In [ ]:
import textwrap

def _escape_triple_quotes(s: str) -> str:
    return s.replace('"""', r'\"\"\"')

def build_pytest_from_samples(samples: dict) -> str:
    # samples is expected to have keys like "input" and "output" (lists of strings). [web:16]
    inputs  = samples.get("input", [])
    outputs = samples.get("output", [])
    n = min(len(inputs), len(outputs))

    cases = []
    for i in range(n):
        inp = _escape_triple_quotes(str(inputs[i]))
        out = _escape_triple_quotes(str(outputs[i]))
        cases.append((inp, out))

    return textwrap.dedent(f"""
    import subprocess
    import sys
    import textwrap
    import pytest

    # Tests assume the solution is saved as solution.py in the project root.
    # If you use a different path, change this constant.
    SOLUTION_PATH = "solution.py"

    @pytest.mark.parametrize("stdin, expected_stdout", {cases!r})
    def test_samples(stdin, expected_stdout):
        p = subprocess.run(
            [sys.executable, SOLUTION_PATH],
            input=stdin,
            text=True,
            capture_output=True,
        )
        assert p.returncode == 0, p.stderr
        assert p.stdout.strip() == expected_stdout.strip()
    """).strip()


In [ ]:
EOS = tokenizer.eos_token

def _escape_triple_quotes(s: str) -> str:
    return s.replace('"""', r'\"\"\"')

def build_pytest_from_samples(samples: dict) -> str:
    inputs  = samples.get("input", [])
    outputs = samples.get("output", [])
    n = min(len(inputs), len(outputs))

    cases = []
    for i in range(n):
        inp = _escape_triple_quotes(str(inputs[i]))
        out = _escape_triple_quotes(str(outputs[i]))
        cases.append((inp, out))

    return textwrap.dedent(f"""
    import subprocess
    import sys
    import pytest

    SOLUTION_PATH = "solution.py"

    @pytest.mark.parametrize("stdin, expected_stdout", {cases!r})
    def test_samples(stdin, expected_stdout):
        p = subprocess.run(
            [sys.executable, SOLUTION_PATH],
            input=stdin,
            text=True,
            capture_output=True,
        )
        assert p.returncode == 0, p.stderr
        assert p.stdout.strip() == expected_stdout.strip()
    """).strip()

def format_example(ex):
    problem = ex.get("description", "")
    code    = ex.get("code", "")
    samples = ex.get("test_samples", {})

    user_prompt = f"""You are given a competitive programming problem and a Python solution script.

Problem:
{problem}

Solution code:

Task:
Write pytest unit tests that validate this script using the provided sample inputs and outputs.
Assume the solution is saved as solution.py and tests will run it as a subprocess.
Return only the test code.
""".strip()

    assistant_answer = build_pytest_from_samples(samples)

    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt},
         {"role": "assistant", "content": assistant_answer}],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text + EOS}

train_dataset = train_ds.map(
    format_example,
    remove_columns=train_ds.column_names,
    num_proc=1,  # or remove this argument
    # batched=True,  # (optional) if your function supports batching
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model=model,
    r=16,
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)

model.gradient_checkpointing_enable()


Unsloth 2025.12.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
print(train_dataset.column_names)
print(train_dataset[0]["text"][:300])


['text']
<|user|>
You are given a competitive programming problem and a Python solution script.

Problem:
There are n persons who initially don't know each other. On each morning, two of them, who were not friends before, become friends.

We want to plan a trip for every evening of m days. On each trip, you 


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,            # use the mapped dataset with ['text'] [file:317]
    dataset_text_field="text",
    formatting_func=lambda ex: ex["text"],  # required by Unsloth in this case [web:349]
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=16):   6%|▌         | 508/8139 [00:00<?, ? examples/s]

In [ ]:
# Train the model
trainer_stats = trainer.train()

In [ ]:
from unsloth import FastLanguageModel

# Enable native faster inference (same as your notebook)
FastLanguageModel.for_inference(model)  # [file:317]

# Fill these with your own content
problem_text = """Given an integer n, print n squared.
Input: a single integer n.
Output: n*n.
"""

solution_code = """n = int(input().strip())
print(n * n)
"""


user_prompt = f"""You are given a competitive programming problem and a Python solution script.

Problem:
{problem_text}

Solution code:
{solution_code}

Task:
Write pytest unit tests that validate this script.
Assume the solution is saved as solution.py and tests will run it as a subprocess.
Return only the test code.
""".strip()

messages = [{"role": "user", "content": user_prompt}]

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")  # [file:317]

# Build an attention mask explicitly (helps when pad token equals eos token)
attention_mask = input_ids.ne(tokenizer.pad_token_id).long()

outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.2,
    do_sample=True,
    top_p=0.9,
)

# Decode only the newly generated tokens (assistant output)
gen_only = outputs[0][input_ids.shape[-1]:]
test_code = tokenizer.decode(gen_only, skip_special_tokens=True)
print(test_code)


In [ ]:
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

In [ ]:
from google.colab import drive
import shutil

# Mount your Google Drive
drive.mount('/content/drive')

# Create a folder in Drive to store the model
drive_path = "/content/drive/MyDrive/gguf_models"
os.makedirs(drive_path, exist_ok=True)

# Move the GGUF file to Drive
gguf_files = [f for f in os.listdir("gguf_model") if f.endswith(".gguf")]
for gguf_file in gguf_files:
    shutil.move(os.path.join("gguf_model", gguf_file), drive_path)
    print(f"Moved {gguf_file} to Google Drive.")
